# Open Notebook in Colab

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/tutorial.ipynb)


Please cite this if you use llm_tracker: <br>
Low, D.M., Mair, P., Nock, M., & Ghosh, S. (2026). Text Psychometrics: Assessing Psychological Constructs in Text Using Natural Language Processing. PLOS Digital Health.


# LLM Tracker: a researcher’s guide

Find psychological constructs in text, then compare the LLM’s coding with human coding.

**How to use this notebook:** run cells from top to bottom with the play button or **Shift+Enter**. Only edit the settings marked below. If a cell shows an error, stop and resolve it before continuing.

**Before you start**
- Use only data approved for sharing with your model provider. Remove identifying information and check your institution’s requirements and the provider’s data-retention policy. See [here](https://github.com/childmindresearch/llm_tracker/blob/main/tutorials/anonymization_tutorial.ipynb) for a guide to anonymization.
- You need an [OpenRouter API key](https://openrouter.ai/). LLM coding and quote matching cost money; test on a few documents first. Never share your key or save it in this notebook.
- LLM coding benefits from comparison with human codes (see section 3). Agreement with human coding is evidence of performance on your dataset, not proof of clinical validity.

# About Construct Detection

**LLM Tracker** is a Python package for identifying instances of psychological
constructs in qualitative text data using large language models.

A psychological construct is a concept describing an internal psychological state, trait, process, capacity, experience, or pattern of relating that can be inferred from what a person says, feels, thinks, perceives, or does. In the following example, NLP is used to detect the constructs of loneliness and anxiety in a sentence:


<img src="https://raw.githubusercontent.com/childmindresearch/llm_tracker/main/tutorials/construct_demo.png" alt="Construct demo" width="400">

## Workflow

1. **Input files** — provide your documents and construct codebook.
2. **Run LLM** — identify constructs in your documents.
3. **Load human coding** — add human-coded data. Data types (long, wide) are described in Step 3.
4. **Compute reliability** — evaluate agreement with human/ground truth codes.
5. **Compare models** — summarize results from multiple models in one table.

<img src="https://raw.githubusercontent.com/childmindresearch/llm_tracker/main/tutorials/workflow.png" alt="workflow" width="300">

## Setup (once)

You only need this notebook and an internet connection. Run the next cell to
download LLM Tracker and the practice files, install the package and
TextDescriptives, and download the English language model used below. You do
not need to clone or download the repository beforehand.

This works in Google Colab or a local Jupyter notebook using Python 3.11–3.13.
The first setup may take a few minutes. If asked to restart the session/kernel,
restart it and rerun this setup cell before continuing. Setup does not call an
LLM or spend API credits.


In [1]:
from pathlib import Path
import subprocess
import sys
import tempfile
from urllib.request import urlretrieve
from zipfile import ZipFile

# Use an existing repository if available; otherwise download it automatically.
project_dir = Path.cwd()
if not (project_dir / "src" / "llm_tracker").is_dir():
    project_dir = project_dir.parent
if not (project_dir / "src" / "llm_tracker").is_dir():
    project_dir = Path.cwd() / "llm_tracker-main"
    if not (project_dir / "src" / "llm_tracker").is_dir():
        print("Downloading LLM Tracker and the practice files...")
        with tempfile.TemporaryDirectory() as temporary_folder:
            archive = Path(temporary_folder) / "llm_tracker.zip"
            urlretrieve("https://codeload.github.com/childmindresearch/llm_tracker/zip/refs/heads/main", archive)
            with ZipFile(archive) as downloaded:
                downloaded.extractall(project_dir.parent)

%pip install -e "{project_dir}" textdescriptives

import spacy
if not spacy.util.is_package("en_core_web_sm"):
    subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm"], check=True)

print("Setup complete. Continue to Input files below.")


Obtaining file:///Users/freymon.perez/Projects/GitHub/llm_tracker
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llm-tracker (pyproject.toml) ... done
  Created wheel for llm-tracker: filename=llm_tracker-0.1.0b1-py3-none-any.whl size=5104 sha256=54efe992abf49479c3e8bf781e7706c7c94b4f6cd01765721e74d4f753e2a932
  Stored in directory: /private/var/folders/09/drdv5jw504142vpxfq711jf815q1qc/T/pip-ephem-wheel-cache-zbgx8zcz/wheels/9d/32/98/c6659aeac80211247bbac02256fee32c44b011d8ded6f9683f
Successfully built llm-tracker
  Attempting uninstall: llm-tracker
    Found existing installation: llm-tracker 0.1.0b1
    Uninstalling llm-tracker-0.1.0b1:
      Successfully uninstalled llm-tracker-0.1.0b1

[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: y

## 1. Input files

Choose for files:
- **Text:** a CSV with one document per row, or a folder of `.txt` files.
- **Codebook:** a JSON file listing construct names and definitions (see [example](https://github.com/childmindresearch/llm_tracker/blob/main/sample_data/tutorial_practice/codebook.json)). You can ask an LLM/chatbot to convert your codebook into JSON.
- **Human coding** (for comparison): coded excerpts or presence/absence ratings for the same documents.

In [2]:
import os
from getpass import getpass

import pandas as pd
from pathlib import Path

from IPython.display import display

from llm_tracker import AnalyzerConfig, LLMTrackerAnalyzer
from llm_tracker.comparison import (
    LLMTrackerComparer,
    build_presence_grid,
    compute_binary_metrics,
    compute_summary_tables,
    format_concatenated,
)
from llm_tracker.file_handlers import (
    codebook_constructs,
    get_document_files,
    load_document,
    load_codebook,
    load_human_coding,
    load_human_wide,
    validate_constructs,
)
from llm_tracker.presence import compare_presence, rank_constructs_by_percentage
from llm_tracker import RunOutput
from llm_tracker.utils import format_coding_table


/Users/freymon.perez/Projects/GitHub/llm_tracker/src/llm_tracker/models.py:40: UserWarning: Field name "construct" in "ConstructInstance" shadows an attribute in parent "BaseModel"
  class ConstructInstance(BaseModel):


Run the next cell. It uses `OPENROUTER_API_KEY` if you already set it; otherwise
paste your API key when asked. Your entry is hidden. Do not print the key or share
an environment file containing it.


In [ ]:
api_key = ""
if not api_key:
    raise ValueError("Enter your OpenRouter API key.")


The settings cell defaults to the bundled practice sample in `sample_data/tutorial_practice/`, so you can run every section without your own data. For your own study, replace the example paths and column names with yours. Use full file paths. For CSV input, document IDs should be unique and match your human coding; `None` uses row numbers starting at 0. For `.txt` files, IDs are filenames without `.txt`.


Compare latest models and costs at [openrouter.ai/models](https://openrouter.ai/models). Copy your model's ID and paste it after `llm_model`.

Set **`output_root` once** below. Every later folder and filename is created from
it, including text descriptives, LLM encodings, comparisons, codebook optimization,
and a shared `run_metadata.csv` recording the runs. Keys are not included in that
metadata. Model names and timestamps are added to generated output names.


In [5]:
# EDIT these settings.
# The defaults point at the bundled practice sample so you can run the whole
# tutorial end to end. Replace them with your own study files for real use.
sample_dir = project_dir / "sample_data" / "tutorial_practice"
input_data = sample_dir / "documents.csv"  # Or a folder of .txt files.
codebook_path = sample_dir / "codebook.json"
text_column = "text"
id_column = "document_id"  # Or None for CSV row numbers.
llm_model = None  # Uses the package default; or enter your OpenRouter model ID.

# This is the only output folder you need to set.
output_root = project_dir / "research_results"

run_text_descriptives = True
max_retries = 2  # Extra attempts after an ordinary coding or quote-matching failure.

# "ceiling": either score 1 or 2 counts as present; "floor": only score 2 counts.
rounding_method = "ceiling"
summary_min_count = 10  # Long-form second summary: include TP + FP + FN >= this count.
display_metrics = None  # All default long-form metrics; or e.g. ["f1", "jaccard"].


### 1.1 Text descriptives (optional)

Before coding, look at the documents themselves: how many there are, how long
they are, and their sentence and vocabulary patterns. This runs locally and
makes **no LLM calls**. Set `run_text_descriptives=False` to skip it.

The displayed table summarizes your corpus; a second CSV saves the measurements
for each document. Both save directly under `output_root`, with a timestamp.
The English model below is suitable for the bundled English example; other
languages need a matching spaCy model.


In [6]:
source_path = Path(input_data)
if not source_path.exists():
    raise FileNotFoundError(f"Check input_data: {source_path}")

if run_text_descriptives:
    from llm_tracker.corpus_summary import summarize_corpus

    if source_path.is_dir():
        documents = [load_document(path)[0] for path in get_document_files(source_path)]
    else:
        input_frame = pd.read_csv(source_path)
        if text_column not in input_frame:
            raise ValueError(f"Check text_column: '{text_column}' was not found in the input CSV.")
        if input_frame[text_column].isna().any():
            raise ValueError("The input CSV contains missing document text.")
        documents = input_frame[text_column].astype(str).tolist()

    text_summary, text_by_document = summarize_corpus(
        documents,
        spacy_model="en_core_web_sm",
        include_readability=True,
        include_sentence_complexity=True,
        include_coherence_sbert=False,
        output_dir=output_root,
        input_path=str(source_path),
    )
    display(text_summary)


Extracting TextDescriptives metrics (descriptive_stats, pos_proportions, readability, dependency_distance)…


/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Computing parse-tree depths…


/Users/freymon.perez/Projects/GitHub/llm_tracker/.venv/lib/python3.12/site-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.5). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


,Metric,Value
0,N (documents),4
1,Sentences per document,1.25 (0.50) [1.00–2.00]
2,Sentence length (words),13.38 (2.14) [11.00–16.00]
3,Tokens per document,16.50 (6.03) [11.00–25.00]
4,Unique tokens per document,14.75 (3.86) [11.00–20.00]
5,Type-token ratio (TTR),0.92 (0.08) [0.80–1.00]
6,Characters per document,67.75 (20.61) [51.00–97.00]
7,1st-person pronoun proportion,22.15% (3.04%) [18.18%–25.00%]
8,Proportion NOUN,18.21% (4.91%) [13.33%–25.00%]
9,Proportion VERB,22.44% (4.43%) [16.67%–26.67%]


## 2. Run LLM coding

This sends each document and your codebook to the model. Results are saved
automatically in a dated folder inside `output_root`. The folder's `DF`
subfolder contains a CSV of the extracted quotes.

Review the extracted quotes below. **Do not continue to comparison if any
documents failed.** The dated folder's `errors` subfolder records failures;
resolve them before rerunning. Rerunning this cell processes all documents
again and incurs new charges.


In [7]:
source_path = Path(input_data)
if not source_path.exists():
    raise FileNotFoundError(f"Check input_data: {source_path}")
codebook = load_codebook(codebook_path)
config_kwargs = dict(
    api_key=api_key, temperature=0, max_retries=max_retries,
    master_output_dir=output_root,
)
if llm_model is not None:
    config_kwargs["model_name"] = llm_model
config = AnalyzerConfig(**config_kwargs)
analyzer = LLMTrackerAnalyzer(config=config)

if source_path.is_dir():
    results_llm, metadata_llm, errors_llm = analyzer.analyze_directory(
        input_dir=input_data,
        codebook_path=codebook_path,
    )
else:
    results_llm, metadata_llm, errors_llm = analyzer.analyze_csv(
        csv_path=input_data,
        text_column=text_column,
        id_column=id_column,
        codebook_path=codebook_path,
    )

encoding_folder = Path(analyzer.output_path)
print("Encoding saved to:", encoding_folder.resolve())
if errors_llm:
    display(errors_llm)
    raise RuntimeError("Some documents failed. Resolve the errors before comparison.")
if not results_llm:
    raise ValueError("No documents were coded. Check your input files.")

display(format_coding_table(results_llm))


Processing [1/4]: doc01
  OK: Found 3 construct instances
  Progress: Successful 1/4; Errors 0/4
Processing [2/4]: doc02
  OK: Found 1 construct instances
  Progress: Successful 2/4; Errors 0/4
Processing [3/4]: doc03
  OK: Found 1 construct instances
  Progress: Successful 3/4; Errors 0/4
Processing [4/4]: doc04
  OK: Found 1 construct instances
  Progress: Successful 4/4; Errors 0/4

Analysis complete!
Output directory: /Users/freymon.perez/Projects/GitHub/llm_tracker/research_results/llm_encodings_google_gemini_3_flash_preview_2026_11_10_100800_904925
Successful: 4/4
Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the s

,doc_id,construct,quote,speaker_id,quote_index,confidence
0,doc01,anxiety,I feel worried about my exams.,None,0:30,2
1,doc01,social_support,My sister listens when I need to talk,None,31:68,2
2,doc01,self_management,I take a walk each evening to manage my stress.,None,74:121,2
3,doc02,anxiety,I worry every night and I cannot sleep because...,None,0:68,2
4,doc03,social_support,My friend helped me carry my groceries when I ...,None,0:82,2
5,doc04,self_management,I started journaling every morning to keep tra...,None,0:61,2


## 3. Load human coding (optional)

**If you only need LLM-coded quotes, stop here.** To compare your LLM results against human-coded data, choose one of the three options below.

| Set `human_source` to | Use this when… |
| --- | --- |
| `synthetic` | You have no human data and want to practice the workflow with the bundled sample. |
| `long` | Your file has one row per coded passage, such as a Dedoose export. |
| `wide` | Your file has one row per document and a column for each construct. |

Read the option that fits your data, then edit the shared settings and run the loading cell. You do not need to prepare all three options.

### Option A: Synthetic human coding (practice only)

Set `human_source = "synthetic"`. The loading cell reads a small reference file that ships with this tutorial (`sample_data/tutorial_practice/synthetic_human_coding.csv`), matched to the bundled practice documents and codebook. **No second LLM run and no human-coded file are needed, and it makes no extra API calls.**

We wrote this reference as synthetic example data. It is **not** real human coding and does not measure agreement with humans; it only gives you something to compare against while you learn the workflow. For any real analysis, load your own human coding as ground truth using **Option B** (long) or **Option C** (wide) below.

### Option B: Long-format human coding (one row per passage)

Set `human_source = "long"`. A document can appear on several rows because the human highlighted several passages:

| document_id | quote | construct |
| --- | --- | --- |
| doc01 | I feel worried about exams. | anxiety |
| doc01 | My sister helps me cope. | social_support |
| doc02 | I worry every night. | anxiety |

Set the ID, quote, and construct column names to match your file. For Dedoose exports these are often `Media Title`, `Excerpt Copy`, and `Codes Applied Combined`. CSV and TSV files are supported; Excel requires the appropriate pandas Excel reader.

- If one cell contains multiple constructs, set `code_separator` to the separator used in your file. Use `;` rather than commas to separate if construct names contain commas. For one construct per row, choose a separator that never occurs in the names.
- Character positions are optional: leave `human_range_column = None` if unavailable. Otherwise set the column name and choose `"dash"` for `858-1159` or `"colon"` for `858:1159`.
- Document IDs must match the LLM input IDs. A file of passages usually omits documents where no constructs were found. Missing documents are **not** automatically treated as confirmed absences; use complete wide-format coding if this applies.

### Option C: Wide-format human coding (one row per document)

Set `human_source = "wide"`. Each construct has its own column: **0 = absent, 1 = present**.

| document_id | text | anxiety_score | social_support_score |
| --- | --- | --- | --- |
| doc01 | I feel worried. My sister helps me cope. | 1 | 1 |
| doc02 | I worry every night. | 1 | 0 |
| doc03 | The library opens at nine. | 0 | 0 |

Set `human_id_column` and `human_text_column` to your column names. The loader removes `_score` from column names, so `anxiety_score` becomes `anxiety`. If your columns use another ending, change `construct_suffix`.

Include a column for **every codebook construct** and complete numeric ratings for every document, including all-zero rows. Blank cells mean missing coding, not absence. `include_empty=True` in the loading cell keeps the all-zero documents.

Wide data records whether a construct appeared, not which passage or how many times. Use **binary presence comparison**, not quote matching or instance-count agreement. Keep `human_presence_threshold = 1` for 0/1 coding; change it only if your study uses a different rating rule (for example, 0.5 for averaged human ratings).

### Choose your source and comparison

Edit only the settings for your selected source; unused file settings are ignored.

- **`quotes`** extracts instances of phrases that express the construct (we later use a paid LLM to match the extracted phrases from the LLM and human-highlighted phrases).
- **`binary_presence`** detects whether each construct appears at least once anywhere in each document (no LLM call needed to compare with human results). Limitation: an LLM finding only one of multiple instances found by a human of a construct gets perfect reliability.

In [8]:
human_source = "synthetic" # EDIT: choose "synthetic", "long", or "wide".
comparison_level = "quotes"  # "binary_presence" must be used for wide data.

# For the synthetic practice option (bundled example data, not real human coding):
synthetic_human_path = str(sample_dir / "synthetic_human_coding.csv")

# For real human data (long or wide):
human_path = "/path/to/human_coding.csv"
human_id_column = "document_id"

# For long format only:
human_quote_column = "quote"
human_construct_column = "construct"
code_separator = ";"
human_range_column = None
human_range_format = "dash"

# For wide format only:
human_text_column = "text"
construct_suffix = "_score"
human_presence_threshold = 1
construct_cols = None  # Wide only: explicit score-column names, or None for all suffix-matching columns.


In [9]:
if comparison_level not in {"quotes", "binary_presence"}:
    raise ValueError('Set comparison_level to "quotes" or "binary_presence".')
if errors_llm or not results_llm:
    raise RuntimeError("Complete LLM coding successfully before loading comparison data.")


if human_source == "synthetic":
    human_results = load_human_coding(
        synthetic_human_path,
        doc_id_col=human_id_column,
        quote_col=human_quote_column,
        construct_col=human_construct_column,
        range_col=None,
        construct_separator=code_separator,
    )
    human_errors = []
    print("PRACTICE ONLY: this reference coding is synthetic example data we wrote, not real human judgments.")
elif human_source == "long":
    human_results = load_human_coding(
        human_path,
        doc_id_col=human_id_column,
        quote_col=human_quote_column,
        construct_col=human_construct_column,
        range_col=human_range_column,
        range_format=human_range_format,
        construct_separator=code_separator,
    )
elif human_source == "wide":
    if comparison_level != "binary_presence":
        raise ValueError('For wide data, set comparison_level to "binary_presence".')
    human_results = load_human_wide(
        human_path,
        doc_id_col=human_id_column,
        text_col=human_text_column,
        construct_suffix=construct_suffix,
        construct_cols=construct_cols,
        presence_threshold=human_presence_threshold,
        include_empty=True,
    )
else:
    raise ValueError('Set human_source to "synthetic", "long", or "wide".')

missing_ids = sorted(set(results_llm) - set(human_results))
if missing_ids:
    raise ValueError(f"Missing human coding for document IDs: {missing_ids}. Check IDs and coding completeness.")

human_results = {doc_id: human_results[doc_id] for doc_id in results_llm}
print(f"Reference coding loaded for all {len(results_llm)} LLM-coded documents.")
display(format_coding_table(human_results))


PRACTICE ONLY: this reference coding is synthetic example data we wrote, not real human judgments.
Reference coding loaded for all 4 LLM-coded documents.
Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the source text
  quote_index : character-level start:end indices of the quote
  confidence  : LLM confidence score (1=indirect or unclear evidence,
                2=clear; missing for human codings)
Explicit classification absences may appear as confidence 0; the default quote prompt omits them.



,doc_id,construct,quote,speaker_id,quote_index,confidence
0,doc01,anxiety,I feel worried about my exams.,None,None,None
1,doc01,social_support,My sister listens when I need to talk,None,None,None
2,doc01,self_management,I take a walk each evening to manage my stress,None,None,None
3,doc02,anxiety,I worry every night and I cannot sleep because...,None,None,None
4,doc03,social_support,My friend helped me carry my groceries when I ...,None,None,None
5,doc04,self_management,I started journaling every morning to keep tra...,None,None,None


### 3.1 Check and map construct names

Names must match the codebook **exactly**, including capitalization. Run the next cell to list codebook names and flag any unknown names in your human coding.

If names differ but mean the same construct, add a pair to `construct_mapping` below: `"name in human data": "name in codebook"`. Only map equivalent constructs. If names were accidentally split at commas, fix `code_separator` and rerun loading instead of mapping fragments.

In [10]:
from llm_tracker.file_handlers import apply_construct_mapping

print("Construct names in your codebook:")
for name in codebook_constructs(codebook):
    print(f"- {name}")
unknown = validate_constructs(human_results, codebook)
print("Names needing correction:", unknown)


Construct names in your codebook:
- anxiety
- social_support
- self_management
Names needing correction: []


In [11]:
# EDIT if needed; otherwise leave empty. Remove # from your mapping lines.
construct_mapping = {
    # "social anxiety sx": "Social Anxiety",
}

if construct_mapping:
    human_results = apply_construct_mapping(human_results, construct_mapping)
validate_constructs(human_results, codebook, strict=True)
print("Construct names match. You can continue to comparison.")


# Wide ratings must cover the full codebook, even when a column is all zero.
if human_source == "wide":
    suffix = Path(human_path).suffix.lower()
    if suffix in {".xlsx", ".xls"}:
        wide_frame = pd.read_excel(human_path)
    else:
        wide_frame = pd.read_csv(human_path, sep="\t" if suffix == ".tsv" else ",")
    if wide_frame[human_id_column].astype(str).str.strip().duplicated().any():
        raise ValueError("Wide human data must have one row per document ID.")
    selected_columns = construct_cols if construct_cols is not None else [
        name for name in wide_frame.columns if construct_suffix and name.endswith(construct_suffix)
    ]
    wide_names = [name[:-len(construct_suffix)] if construct_suffix and name.endswith(construct_suffix) else name for name in selected_columns]
    wide_names = {construct_mapping.get(name, name) for name in wide_names}
    if wide_names != set(codebook_constructs(codebook)):
        raise ValueError("Wide score columns must cover exactly the codebook constructs. Check construct_cols, the suffix, and the name mapping.")


Construct names match. You can continue to comparison.


## 4. Compute reliability

The next cell runs your selected comparison, using the loaded human/ground truth coding as the reference. Only the LLM-coded documents are evaluated; extra documents in the human file are excluded.

We have included several reliability metrics but also show you how to roll your own in section 4.1 below. 


| Measure | Research question |
| --- | --- |
| Sensitivity (recall) | How much of the human coding did the LLM find? |
| Precision | How much of the LLM coding agreed with human coding? |
| F1 | How well does the LLM balance sensitivity and precision? |
| Jaccard | How much overlap is there between the two sets of detections? |

Higher values generally mean better agreement. Missing/undefined values are not low performing and need to be reviewed. If `comparison_level` = 'quote', comparisons assess coded phrases; if `comparison_level` = 'binary_presence', comparisons assess whole documents, so their scores answer different questions.

Inspect disagreements, not just summary scores. Presence tables also include agreement on absence; many absent constructs can make accuracy look high even when detection is poor.

The default LLM prompt uses score **1** for indirect/unclear evidence and **2**
for clear evidence. With `rounding_method="ceiling"`, either score counts as a
detection. With `"floor"`, only score 2 counts. The same rule applies to both
comparison types. Human coding and original saved LLM scores are unchanged.


### Which metrics each `comparison_level` reports

Both levels report **sensitivity**, **precision**, **F1**, and **Jaccard** (from TP/FP/FN), plus these extras:

**`quotes`** (span level, the saved formatted quote-summary CSV):

| Column | Meaning |
| --- | --- |
| `tp`, `fp`, `fn` | Span-level match counts |
| `pr_auc` | Precision–recall summary using LLM confidence |
| `interviews_with_construct` | Documents containing each construct |

No `tn`: a construct neither coder quoted produces no span.

**`binary_presence`** (document level, the saved `metrics_llm_human_binary_...csv`):

| Column | Meaning |
| --- | --- |
| `n_docs`, `human_present`, `llm_present` | Documents rated and marked present |
| `tp`, `fp`, `fn`, `tn` | Full confusion matrix |
| `pabak`, `cohens_kappa`, `kripp_alpha_nominal` | Chance-corrected agreement |

**Why kappa is binary-only:** kappa's chance term needs all four cells, including TN. The `quotes` level has no TN, so kappa, PABAK, and Krippendorff's alpha are omitted. Jaccard appears at both levels because it ignores TN: `TP / (TP + FP + FN)`.

**The two long-form Overall rows:** the first describes scores across all
constructs as mean (SD) [p5, p25, p50, p75, p95]. Each construct has equal weight.
The second uses only constructs with `tp + fp + fn >= summary_min_count` (default
10). This is a count cutoff, not a Jaccard cutoff; it does not remove any construct
rows. The small practice dataset may have no qualifying constructs. These summary
distributions are different from pooling all matches into one score. Short-form
binary metrics still have one pooled Overall row.

`display_metrics` selects which existing long-form metrics are displayed. Leave
it `None` for all defaults; it does not add new metrics or change the calculations.


In [12]:
if errors_llm or not results_llm:
    raise RuntimeError("Complete LLM coding successfully before comparing.")
if comparison_level not in {"quotes", "binary_presence"}:
    raise ValueError('Set comparison_level to "quotes" or "binary_presence".')
if human_source == "wide" and comparison_level != "binary_presence":
    raise ValueError('Wide data requires comparison_level = "binary_presence".')
validate_constructs(human_results, codebook, strict=True)

print(f"Reference source: {human_source}")
if comparison_level == "binary_presence":
    comparison_table = compare_presence(
        human_results=human_results, llm_results=results_llm,
        constructs=list(codebook_constructs(codebook)), document_ids=list(results_llm),
        master_output_dir=output_root, model_name=config.model_name,
        rounding_method=rounding_method, binarize_confidence=True,
    )
    summary = compute_binary_metrics(comparison_table)
    summary_folder = Path(comparison_table.attrs["output_path"])
    comparison_run = RunOutput.load(summary_folder)
    model_summary_path = summary_folder / comparison_run.filename("doc_binary")
    review_scores = summary
else:
    comparator = LLMTrackerComparer(config=config)
    comparison_table = comparator.compare_results(
        human_results=human_results, llm_results=results_llm,
        output_dir=str(output_root), rounding_method=rounding_method,
        binarize_confidence=True,
    )
    summary_folder = Path(comparison_table.attrs["output_path"])
    comparison_run = RunOutput.load(summary_folder)
    per_document, pooled, weighted = compute_summary_tables(comparison_table)
    summary = format_concatenated(pooled, metrics=display_metrics, min_count=summary_min_count)
    comparison_run.table("metrics_llm_human_quote_matching_formatted", summary)
    model_summary_path = summary_folder / comparison_run.filename("span_concatenated")
    review_scores = pooled  # Use numeric scores, not rounded/formatted summary strings.

comparison_run.update(reference_source=human_source, summary_min_count=summary_min_count,
                      display_metrics="all_defaults" if display_metrics is None else display_metrics,
                      human_input_path=str(synthetic_human_path if human_source == "synthetic" else human_path),
                      human_id_column=human_id_column,
                      human_presence_threshold=human_presence_threshold if human_source == "wide" else "N/A")
display(summary)
display(comparison_table.head(20))
print("Results saved under:", summary_folder.resolve())
print("Numeric summary for comparing models:", model_summary_path.resolve())


Reference source: synthetic
Coding Table
------------
One row per construct instance across all documents. Each row represents
a single quote identified by the LLM as an instance of a construct.

Columns:
  doc_id      : document identifier (filename or subreddit_author)
  construct   : psychological construct the instance belongs to
  quote       : exact quote extracted from the source text
  speaker_id  : speaker identifier if available in the source text
  quote_index : character-level start:end indices of the quote
  confidence  : LLM confidence score (1=indirect or unclear evidence,
                2=clear; missing for human codings)
Explicit classification absences may appear as confidence 0; the default quote prompt omits them.



,construct,tp,fp,fn,count,sensitivity,precision,f1,jaccard,pr_auc,interviews_with_construct [p5-p95]
0,anxiety,2,0,0,2,1.00,1.00,1.00,1.00,-,2 [0.00-1.00]
1,self_management,2,0,0,2,1.00,1.00,1.00,1.00,-,2 [0.00-1.00]
2,social_support,2,0,0,2,1.00,1.00,1.00,1.00,-,2 [0.00-1.00]
3,Overall,"2.00 (0.00) [2.00, 2.00, 2.00, 2.00, 2.00]","0.00 (0.00) [0.00, 0.00, 0.00, 0.00, 0.00]","0.00 (0.00) [0.00, 0.00, 0.00, 0.00, 0.00]","2.00 (0.00) [2.00, 2.00, 2.00, 2.00, 2.00]","1.00 (0.00) [1.00, 1.00, 1.00, 1.00, 1.00]","1.00 (0.00) [1.00, 1.00, 1.00, 1.00, 1.00]","1.00 (0.00) [1.00, 1.00, 1.00, 1.00, 1.00]","1.00 (0.00) [1.00, 1.00, 1.00, 1.00, 1.00]",-,
4,Overall (count >= 10; n = 0),-,-,-,-,-,-,-,-,-,


,doc_id,construct,status,human_quote,llm_quote,human_indices,llm_indices,human_confidence,llm_confidence,paraphrase,span_overlap,match_confidence,tp,fp,fn
0,doc01,anxiety,matched,I feel worried about my exams.,I feel worried about my exams.,None,0:30,None,1,False,None,1.0,1,0,0
1,doc01,self_management,matched,I take a walk each evening to manage my stress,I take a walk each evening to manage my stress.,None,74:121,None,1,False,None,1.0,1,0,0
2,doc01,social_support,matched,My sister listens when I need to talk,My sister listens when I need to talk,None,31:68,None,1,False,None,1.0,1,0,0
3,doc02,anxiety,matched,I worry every night and I cannot sleep because...,I worry every night and I cannot sleep because...,None,0:68,None,1,False,None,1.0,1,0,0
4,doc03,social_support,matched,My friend helped me carry my groceries when I ...,My friend helped me carry my groceries when I ...,None,0:82,None,1,True,None,1.0,1,0,0
5,doc04,self_management,matched,I started journaling every morning to keep tra...,I started journaling every morning to keep tra...,None,0:61,None,1,False,None,1.0,1,0,0


Results saved under: /Users/freymon.perez/Projects/GitHub/llm_tracker/research_results/comparison_long_google_gemini_3_flash_preview_2026_11_10_100813_666232
Numeric summary for comparing models: /Users/freymon.perez/Projects/GitHub/llm_tracker/research_results/comparison_long_google_gemini_3_flash_preview_2026_11_10_100813_666232/metrics_llm_human_quote_matching_pooled_google_gemini_3_flash_preview_2026_11_10_100813_666232.csv


### Constructs to review

Here we flag **Jaccard below 0.3** for quote comparisons and **Cohen's kappa below
0.4** for binary presence comparisons. These are the review cutoffs used in this
tutorial. If any constructs are listed:

**These constructs had below moderate reliability with human codes. Consider
removing or noting with an asterisk \*.**

Review the disagreements before deciding what to change. The cell lists and saves
the constructs; it does not remove or rename them. Undefined scores are saved in
a separate list for review, not treated as poor scores. When using the synthetic
practice option, the reference is synthetic rather than real human coding.


In [13]:
review_metric, review_cutoff = ("jaccard", 0.3) if comparison_level == "quotes" else ("cohens_kappa", 0.4)
construct_scores = review_scores.loc[review_scores["construct"] != "Overall"].copy()
values = pd.to_numeric(construct_scores[review_metric], errors="coerce")
low_agreement = construct_scores.loc[values.notna() & (values < review_cutoff), ["construct", review_metric]].copy()
undefined_agreement = construct_scores.loc[values.isna(), ["construct", review_metric]].copy()
comparison_run.table("constructs_below_agreement_review_cutoff", low_agreement)
comparison_run.table("constructs_with_undefined_agreement", undefined_agreement)
comparison_run.update(review_metric=review_metric, review_cutoff=review_cutoff)
display(low_agreement)
if low_agreement.empty:
    print("No constructs fell below this review cutoff.")
else:
    print("These constructs had below moderate reliability with human codes. Consider removing or noting with an asterisk *")
if not undefined_agreement.empty:
    print("These constructs have undefined scores and need separate review:")
    display(undefined_agreement)


,construct,jaccard


No constructs fell below this review cutoff.


### 4.1 Roll your own metrics

LLM Tracker already implements several agreement metrics for you: precision, sensitivity, F1, Jaccard, PABAK, Cohen's kappa, and Krippendorff's alpha for detection, plus ordinal metrics on instance counts. When your study needs something the package does not ship, you can compute it yourself: everything the built-in metrics use is available in an ordinary pandas DataFrame. This subsection shows how, working through two examples: **accuracy** on binary presence/absence and **R²** on ordinal instance counts.

**Which table do you need?** This is the decision that matters most, and getting it wrong is the easiest way to compute a number that looks fine and means nothing.

A `quotes` `comparison_table` is **span-level**: one row per coded instance, labelled `matched`, `human_only`, or `llm_only`. It has no true negatives, because a construct neither coder marked produces no row at all. Any metric whose formula contains TN cannot be computed from it.

The **presence grid** is **document-level**: it crosses every document with every construct, so pairs neither coder marked appear as explicit `0 / 0` rows — the true negatives. When `comparison_level = "binary_presence"`, your `comparison_table` is already this grid. When `comparison_level = "quotes"`, the next cell derives the grid and fills
in the complete document and codebook roster. This keeps documents and constructs
with no detections in the denominator, including for the ranked table at the end.

| Column | Meaning |
|---|---|
| `doc_id`, `construct` | The cell being rated |
| `human_count`, `llm_count` | How many instances each coder marked |
| `human_present`, `llm_present` | The counts binarized at the presence threshold |

Accuracy needs TN, so it reads the grid. R² on counts also needs the zero cells — a document where the LLM correctly found nothing is real information about its calibration — so it reads the grid too.


In [14]:
if comparison_level == "binary_presence":
    presence_grid = comparison_table
else:
    # First derive counts from quote matches, then include every completed
    # document and every codebook construct, even when neither coder found it.
    observed_grid = build_presence_grid(comparison_table)
    full_roster = pd.MultiIndex.from_product(
        [list(results_llm), list(codebook_constructs(codebook))], names=["doc_id", "construct"],
    )
    presence_grid = observed_grid.set_index(["doc_id", "construct"]).reindex(full_roster, fill_value=0).reset_index()
    count_columns = ["human_count", "llm_count", "human_present", "llm_present"]
    presence_grid[count_columns] = presence_grid[count_columns].astype(int)
    presence_grid.attrs.update(comparison_table.attrs)
    comparison_run.table("presence_grid", presence_grid)
    binary_document_summary = compute_binary_metrics(presence_grid)

display(presence_grid.head(10))


,doc_id,construct,human_count,llm_count,human_present,llm_present
0,doc01,anxiety,1,1,1,1
1,doc01,social_support,1,1,1,1
2,doc01,self_management,1,1,1,1
3,doc02,anxiety,1,1,1,1
4,doc02,social_support,0,0,0,0
5,doc02,self_management,0,0,0,0
6,doc03,anxiety,0,0,0,0
7,doc03,social_support,1,1,1,1
8,doc03,self_management,0,0,0,0
9,doc04,anxiety,0,0,0,0


Accuracy is the proportion of document × construct cells where the two coders reach the same present/absent verdict: `(TP + TN) / total`. On the grid that is an elementwise comparison of the two presence columns.

R² uses the counts themselves instead of discarding them to presence, asking how much of the variance in the human's counts the LLM's counts account for: `1 - SS_residual / SS_total`, with the human counts as the reference.

Both helpers return `NaN` rather than a number where the statistic is undefined. This mirrors what the package does internally in `_safe_kappa` and `_safe_alpha`: a construct the human never coded has zero variance in its counts, and silently reporting `0.0` there would look like a real, poor score instead of an absent one.

In [15]:
import numpy as np
import pandas as pd


def accuracy(human_present, llm_present):
    """Proportion of (document, construct) cells where the coders agree."""
    if len(human_present) == 0:
        return np.nan
    return float((np.asarray(human_present) == np.asarray(llm_present)).mean())


def r2(human_counts, llm_counts):
    """R^2 of the LLM's instance counts against the human's."""
    human_counts = np.asarray(human_counts, dtype=float)
    llm_counts = np.asarray(llm_counts, dtype=float)
    if len(human_counts) < 2:
        return np.nan
    ss_total = ((human_counts - human_counts.mean()) ** 2).sum()
    if ss_total == 0:  # human never varied — nothing to explain
        return np.nan
    ss_residual = ((human_counts - llm_counts) ** 2).sum()
    return float(1 - ss_residual / ss_total)


def custom_metrics(grid):
    """Accuracy and R^2 per construct, plus a pooled Overall row."""
    groups = [(str(name), g) for name, g in grid.groupby("construct")]
    groups.append(("Overall", grid))

    rows = []
    for construct, g in groups:
        rows.append(
            {
                "construct": construct,
                "n_cells": len(g),
                "human_present": int(g["human_present"].sum()),
                "llm_present": int(g["llm_present"].sum()),
                "accuracy": round(
                    accuracy(g["human_present"], g["llm_present"]), 4
                ),
                "r2_counts": round(r2(g["human_count"], g["llm_count"]), 4),
            }
        )
    return pd.DataFrame(rows)


custom_summary = custom_metrics(presence_grid)
display(custom_summary)
comparison_run.table("metrics_llm_human_custom", custom_summary)


,construct,n_cells,human_present,llm_present,accuracy,r2_counts
0,anxiety,4,2,2,1.0,1.0
1,self_management,4,2,2,1.0,1.0
2,social_support,4,2,2,1.0,1.0
3,Overall,12,6,6,1.0,1.0


PosixPath('/Users/freymon.perez/Projects/GitHub/llm_tracker/research_results/comparison_long_google_gemini_3_flash_preview_2026_11_10_100813_666232/metrics_llm_human_custom_google_gemini_3_flash_preview_2026_11_10_100813_666232.csv')

## 5. Compare results from different models (optional)

Run each model on the **same documents, human coding, codebook, and comparison settings** first. Then enter the numeric summary paths printed at the end of Step 4, with a label
for each model. Use the same comparison type for every model. For quote runs,
choose `metrics_llm_human_quote_matching_pooled_...csv`; for binary runs, choose
`metrics_llm_human_binary_...csv`. These contain numbers, not formatted summary text.

This section reads those files only: it makes **no API calls** once the setup and output settings are available; no encoding rerun is needed.

The displayed and saved table contains **one row per model**, summarizing all construct rows except the pre-existing `Overall` row.

Scores are summarized **across constructs**, not across documents: `mean (SD) [p5, p25, p50, p75, p95]; n=valid scores`. Each construct has equal weight. These are descriptive percentiles, not confidence intervals, and are different from pooled performance. Counts show `total;` followed by that distribution. Undefined scores (`-` or blank in the source) are excluded, not treated as zero; `n` shows how many remain. SD is unavailable for a single value.

The table also lists constructs above, below, or with an undefined review score:
Jaccard 0.3 for quote runs, or Cohen's kappa 0.4 for binary runs. The optional
`model_comparison_min_count` setting restricts the across-construct summaries to
`tp + fp + fn` at or above that value; `None` includes all. Source CSVs are never
modified. Results save under `output_root` with model/timestamp naming and metadata.

Leave `run_model_comparison=False` to skip this section and continue to the final
ranking table. Only turn it on after entering real saved paths.


In [16]:
# EDIT only if comparing saved results from multiple models.
run_model_comparison = False
summary_csv_paths = []  # e.g. ["/path/to/model_A/numeric_summary.csv", "/path/to/model_B/numeric_summary.csv"]
model_labels = []  # One label per path, e.g. ["Model A", "Model B"].
saved_comparison_type = "quotes"  # "quotes" or "binary_presence"; all inputs must match.
model_comparison_min_count = None  # Or 10 to use only constructs meeting that count cutoff.


In [17]:
import numpy as np

def describe_values(values):
    values = values.dropna()
    if values.empty:
        return "-; n=0"
    percentiles = ", ".join(f"{value:.2f}" for value in values.quantile([0.05, 0.25, 0.5, 0.75, 0.95]))
    sd = f"{values.std():.2f}" if len(values) > 1 else "-"
    return f"{values.mean():.2f} ({sd}) [{percentiles}]; n={len(values)}"

def summarize_model(frame, model_label, score, cutoff, metric_columns):
    row = {"model": model_label, "n_constructs": len(frame), "constructs_included": "; ".join(frame["construct"])}
    for column in ["tp", "fp", "fn", "count"]:
        total = f"{frame[column].sum():.0f}" if not frame.empty else "0"
        row[column] = f"{total}; {describe_values(frame[column])}"
    for column in metric_columns:
        row[column] = describe_values(frame[column])
    above = frame.loc[frame[score] >= cutoff, "construct"]
    below = frame.loc[frame[score] < cutoff, "construct"]
    undefined = frame.loc[frame[score].isna(), "construct"]
    row.update(review_metric=score, review_cutoff=cutoff,
               constructs_at_or_above_cutoff="; ".join(above), n_at_or_above_cutoff=len(above),
               constructs_below_cutoff="; ".join(below), n_below_cutoff=len(below),
               constructs_undefined="; ".join(undefined), n_undefined=len(undefined))
    return row

if run_model_comparison:
    if saved_comparison_type not in {"quotes", "binary_presence"}:
        raise ValueError("Choose the comparison type used in the saved runs.")
    if not summary_csv_paths or len(summary_csv_paths) != len(model_labels):
        raise ValueError("Enter at least one numeric summary path and a matching model label.")
    if any(not label.strip() for label in model_labels) or len(set(model_labels)) != len(model_labels):
        raise ValueError("Model labels must be nonempty and unique.")
    if model_comparison_min_count is not None and (isinstance(model_comparison_min_count, bool) or not isinstance(model_comparison_min_count, int) or model_comparison_min_count < 0):
        raise ValueError("model_comparison_min_count must be a nonnegative integer or None.")
    metric_columns = ["sensitivity", "precision", "f1", "jaccard"] + (["pr_auc"] if saved_comparison_type == "quotes" else ["pabak", "cohens_kappa", "kripp_alpha_nominal"])
    score, cutoff = ("jaccard", 0.3) if saved_comparison_type == "quotes" else ("cohens_kappa", 0.4)
    comparison_rows = []
    input_paths = [Path(path).expanduser().resolve() for path in summary_csv_paths]
    for model_label, path in zip(model_labels, input_paths, strict=True):
        frame = pd.read_csv(path)
        required_columns = ["construct", "tp", "fp", "fn", *metric_columns]
        if set(required_columns) - set(frame):
            raise ValueError(f"{path}: choose the numeric summary for the selected comparison type.")
        frame = frame.loc[frame["construct"] != "Overall", required_columns].copy()
        if frame.empty or frame["construct"].isna().any() or frame["construct"].duplicated().any():
            raise ValueError(f"{path}: expected unique, nonempty construct rows.")
        for column in required_columns[1:]:
            frame[column] = pd.to_numeric(frame[column], errors="raise")
            values = frame[column].dropna()
            if not np.isfinite(values).all():
                raise ValueError(f"{path}: {column} contains infinite values.")
            if column in {"tp", "fp", "fn"}:
                if frame[column].isna().any() or (values < 0).any() or (values % 1 != 0).any():
                    raise ValueError(f"{path}: invalid counts in {column}.")
            elif column == "kripp_alpha_nominal":
                if (values > 1).any():
                    raise ValueError(f"{path}: alpha cannot exceed 1.")
            elif not values.between(-1 if column in {"cohens_kappa", "pabak"} else 0, 1).all():
                raise ValueError(f"{path}: values outside the expected range in {column}.")
        frame["count"] = frame[["tp", "fp", "fn"]].sum(axis=1)
        if model_comparison_min_count is not None:
            frame = frame.loc[frame["count"] >= model_comparison_min_count].copy()
        row = summarize_model(frame, model_label, score, cutoff, metric_columns)
        row["source_csv"] = str(path)
        comparison_rows.append(row)
    model_comparison = pd.DataFrame(comparison_rows)
    display(model_comparison)
    model_comparison_run = RunOutput.create(
        output_root, "comparison_long" if saved_comparison_type == "quotes" else "comparison_short",
        "multiple_models", mode="model_comparison", source_summary_paths=[str(path) for path in input_paths],
        min_count=model_comparison_min_count, review_metric=score, review_cutoff=cutoff,
    )
    comparison_csv_path = model_comparison_run.table("model_summary_comparison", model_comparison)
    model_comparison_run.finish()
    print("Saved model comparison to:", comparison_csv_path)


## 6. Rank constructs by document percentage

This final table answers: **in what percentage of compared documents did the LLM
detect each construct?** It works for both quote and binary-presence comparisons.
It makes no LLM calls and saves automatically beside the comparison outputs.

For example, **26.2% (28/107)** means the LLM detected that construct in 28 of the
107 compared documents. Five quotes in one document still count as one document.
The denominator includes all compared documents, even when neither coder detected
anything. This is **not** the percentage of quotes or agreement with humans.

The table uses the floor/ceiling decision already applied during comparison.
It includes zero-detection constructs, has no Overall row, and sorts tied counts
alphabetically. Rank is the row number, so tied percentages have consecutive ranks.


In [18]:
if comparison_level == "quotes":
    print("Agreement summary, including the count-filtered Overall row:")
    display(summary)
ranked_constructs = rank_constructs_by_percentage(presence_grid)
display(ranked_constructs)
print("Ranked table saved to:", ranked_constructs.attrs["csv_path"])


Agreement summary, including the count-filtered Overall row:


,construct,tp,fp,fn,count,sensitivity,precision,f1,jaccard,pr_auc,interviews_with_construct [p5-p95]
0,anxiety,2,0,0,2,1.00,1.00,1.00,1.00,-,2 [0.00-1.00]
1,self_management,2,0,0,2,1.00,1.00,1.00,1.00,-,2 [0.00-1.00]
2,social_support,2,0,0,2,1.00,1.00,1.00,1.00,-,2 [0.00-1.00]
3,Overall,"2.00 (0.00) [2.00, 2.00, 2.00, 2.00, 2.00]","0.00 (0.00) [0.00, 0.00, 0.00, 0.00, 0.00]","0.00 (0.00) [0.00, 0.00, 0.00, 0.00, 0.00]","2.00 (0.00) [2.00, 2.00, 2.00, 2.00, 2.00]","1.00 (0.00) [1.00, 1.00, 1.00, 1.00, 1.00]","1.00 (0.00) [1.00, 1.00, 1.00, 1.00, 1.00]","1.00 (0.00) [1.00, 1.00, 1.00, 1.00, 1.00]","1.00 (0.00) [1.00, 1.00, 1.00, 1.00, 1.00]",-,
4,Overall (count >= 10; n = 0),-,-,-,-,-,-,-,-,-,


,Rank,Construct,% (N)
0,1,anxiety,50.0% (2/4)
1,2,self_management,50.0% (2/4)
2,3,social_support,50.0% (2/4)


Ranked table saved to: /Users/freymon.perez/Projects/GitHub/llm_tracker/research_results/comparison_long_google_gemini_3_flash_preview_2026_11_10_100813_666232/llm_construct_presence_ranked_google_gemini_3_flash_preview_2026_11_10_100813_666232.csv


### Where everything was saved

You set one folder at the top: `output_root`. It now contains text-descriptive
CSVs, model/timestamp encoding and comparison folders, and `run_metadata.csv`.
Encoding folders include `DF/` for the quote dataframe. Comparison folders include
quote views, match or presence tables, metrics, review lists, and the final ranking.
You do not need to type timestamped paths into later steps.


In [19]:
print("All study outputs:", Path(output_root).resolve())
print("Encoding run:", encoding_folder.resolve())
print("Comparison run:", summary_folder.resolve())
print("Run metadata:", (Path(output_root) / "run_metadata.csv").resolve())
print("\nComparison CSV files:")
for path in sorted(summary_folder.glob("*.csv")):
    print(" ", path.name)


All study outputs: /Users/freymon.perez/Projects/GitHub/llm_tracker/research_results
Encoding run: /Users/freymon.perez/Projects/GitHub/llm_tracker/research_results/llm_encodings_google_gemini_3_flash_preview_2026_11_10_100800_904925
Comparison run: /Users/freymon.perez/Projects/GitHub/llm_tracker/research_results/comparison_long_google_gemini_3_flash_preview_2026_11_10_100813_666232
Run metadata: /Users/freymon.perez/Projects/GitHub/llm_tracker/research_results/run_metadata.csv

Comparison CSV files:
  constructs_below_agreement_review_cutoff_google_gemini_3_flash_preview_2026_11_10_100813_666232.csv
  constructs_with_undefined_agreement_google_gemini_3_flash_preview_2026_11_10_100813_666232.csv
  llm_construct_presence_ranked_google_gemini_3_flash_preview_2026_11_10_100813_666232.csv
  llm_human_presence_grid_google_gemini_3_flash_preview_2026_11_10_100813_666232.csv
  llm_human_quote_matches_google_gemini_3_flash_preview_2026_11_10_100813_666232.csv
  llm_quotes_by_construct_google_